### BASELINE - ALGORITMO GENÉTICO

#### IMPORTS

In [11]:
from baseline.POSSIBLE import DISTRICTS_POINTS as DP
import math
import random
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import shapely.geometry

##### DECLARAÇÃO DE TIPOS E VARIÁVEIS

In [12]:
POPULATION_NUMBER = 1
ITERATIONS = 1
DISASTER_COUNTDOWN_EVERY = 30
PERCENTAGE_OF_DEATHS = 0.5

NUMBER_LOCATIONS = len(DP)
NUMBER_AMBUS_TYPE_A = 1
NUMBER_AMBUS_TYPE_B = 4
LOCATIONS = [] 
DISTANCE_MATRIX = []
RAIO = 0.0084
#10min 15min, 30min, 1h
#0.0056 0.0084, 0.0168, 0.0336



##### MÉTODOS AUXILIARES

In [13]:
def random_number(start, end):
    return random.randrange(start, end)

#### ENTIDADE LOCATION

In [14]:
# Região com lat e long e densidade populacional (peso)
class Location:
    def __init__(self, *args):
        self.id = args[0]
        self.coord_x = args[1]
        self.coord_y = args[2]
        self.weight = args[3]

    def __str__(self):
        return f'({self.id}: {self.coord_x}, {self.coord_y}) - {self.weight}'
    

##### FUNÇÕES REFERENTES A DISTANCIA E LOCALIZAÇÃO

In [15]:
def calc_distance(x1, y1, x2, y2):
    return math.dist((x1, y1), (x2, y2))
    
def prepare_locations():
    keys = list(DP)
    for i in range(NUMBER_LOCATIONS):
        local = Location(
            keys[i],
            DP[keys[i]][0],
            DP[keys[i]][1],
            DP[keys[i]][2]
        )
        LOCATIONS.append(local)

def create_distance_matrix():
    for i in range(NUMBER_LOCATIONS):
        distances = []
        for j in range(NUMBER_LOCATIONS):
            distances.append(calc_distance(LOCATIONS[i].coord_x, LOCATIONS[i].coord_y, LOCATIONS[j].coord_x, LOCATIONS[j].coord_y))
        DISTANCE_MATRIX.append(distances)

#### ENTIDADE SOLUTION

In [16]:
class Solution:
    def __init__(self):
        self.xA = [0] * len(DP)
        self.xB = [0] * len(DP)
        self.non_zeros_A = []
        self.non_zeros_B = []
        self.rank = 0
        self.coverage_points = []

    def __lt__(self, other):
        return self.rank < other.rank
    
    def __gt__(self, other):
        return self.rank > other.rank

#### ALGORITMO GENÉTICO

In [17]:
def create_solution():
    new_solution = Solution()

    while len(new_solution.non_zeros_A) < NUMBER_AMBUS_TYPE_A:
        random_index = random_number(0, NUMBER_LOCATIONS)
        if random_index not in new_solution.non_zeros_A:
            new_solution.non_zeros_A.append(random_index)
            new_solution.xA[random_index] = 1

    while len(new_solution.non_zeros_B) < NUMBER_AMBUS_TYPE_B:
        random_index = random_number(0, NUMBER_LOCATIONS)
        if random_index not in new_solution.non_zeros_B:
            new_solution.non_zeros_B.append(random_index)
            new_solution.xB[random_index] = 1

    return new_solution

def evaluate_solution(solution: Solution):
    solution.coverage_points = [0]*NUMBER_LOCATIONS

    for j in range(NUMBER_LOCATIONS):
        for i in range(len(solution.non_zeros_A)):
            if((DISTANCE_MATRIX[solution.non_zeros_A[i]][j] <= RAIO)):
                # print(f"{DISTANCE_MATRIX[solution.non_zeros_A[i]][j]}")
                if(solution.xA[solution.non_zeros_A[i]] == 1):
                    solution.coverage_points[j] = 1
        for i in range(len(solution.non_zeros_B)):
            if(DISTANCE_MATRIX[solution.non_zeros_B[i]][j] <= RAIO):
                # print(f"{DISTANCE_MATRIX[solution.non_zeros_B[i]][j]}")
                if(solution.xB[solution.non_zeros_B[i]] == 1):
                    solution.coverage_points[j] = 1

    for i in range(len(solution.coverage_points)):
        if(LOCATIONS[i].weight != "nan"):
            if(solution.coverage_points[i] == 1):
                solution.rank += float(LOCATIONS[i].weight)
        

def crossover_auxiliar():
    new_value = random_number(0,1)
    if(new_value == 0):
        return 0
    else:
        return 1
    

def crossover(base: Solution, guia:Solution):
    nova_solucao = Solution()

    for i in range(NUMBER_LOCATIONS):
        if(base.xA[i] == guia.xA[i]):
            nova_solucao.xA[i] = (base.xA[i])
        elif(base.xA[i] == 1 and guia.xA[i] == 0):
            nova_solucao.xA[i] = base.xA[i]
        else:
            nova_solucao.xA[i] = crossover_auxiliar()

        if(nova_solucao.xA[i] == 1):
            nova_solucao.non_zeros_A.append(i)
    
        if(base.xB[i] == guia.xB[i]):
            nova_solucao.xB[i] = (base.xB[i])
        elif(base.xB[i] == 1 and guia.xB[i] == 0):
            nova_solucao.xB[i] = base.xB[i]
        else:
            nova_solucao.xB[i] = crossover_auxiliar()

        if(nova_solucao.xB[i] == 1):
            nova_solucao.non_zeros_B.append(i)        
        
    evaluate_solution(nova_solucao)
    return nova_solucao


def choose_parent(population: list[Solution], type_parent):
    temp_pop = []
    sum_rank = 1
    for i in range(POPULATION_NUMBER):   
        sum_rank += population[i].rank  
    for i in range(POPULATION_NUMBER):
        temp_pop.append((population[i].rank)/(sum_rank - 1))   

    rank_benchmark = random.uniform(0, 1)   
    if(type_parent == 1): 
        for i in range(POPULATION_NUMBER):
            if( rank_benchmark < temp_pop[i]):
                return population[i]
    else:
        for i in range(POPULATION_NUMBER):
            if( rank_benchmark > temp_pop[i]):
                return population[i]
    j = random_number(0, POPULATION_NUMBER - 1)        
    return population[j]


def disaster(population: list[Solution]):
    deaths = int(PERCENTAGE_OF_DEATHS * POPULATION_NUMBER)
    for _ in range(deaths):
        next_death = random_number(0, len(population))
        population.pop(next_death)
    for _ in range(deaths):
        population.append(create_solution())
    
    return population



### FUNÇÃO PRINCIPAL

In [22]:
def main():
    random.seed()  # ou random.seed(669)

    prepare_locations()
    create_distance_matrix()

    population = []

    # Criação da população inicial
    for _ in range(POPULATION_NUMBER):
        new_solution = create_solution()
        evaluate_solution(new_solution)
        population.append(new_solution)

    if not population:
        raise ValueError("A população inicial ficou vazia.")

    population.sort()
    best_solution_so_far = population[-1]

    print("iteration 0")
    print(best_solution_so_far.rank)

    for generation in range(ITERATIONS):
        print(f"iteration {generation + 1}")

        if population and best_solution_so_far.rank < population[-1].rank:
            best_solution_so_far = population[-1]
            print(population[-1].rank)

        new_pop = []

        PERCENTAGE_ELITISM = 30
        PERCENTAGE_BAD_SOLUTION = 20
        PERCENTAGE_CHILD_SOLUTIONS = 50

        amount_elite_ind = int((PERCENTAGE_ELITISM * POPULATION_NUMBER) / 100)
        amount_bad_ind = int((PERCENTAGE_BAD_SOLUTION * POPULATION_NUMBER) / 100)
        amount_new_ind = int((PERCENTAGE_CHILD_SOLUTIONS * POPULATION_NUMBER) / 100)

        # 1) Elitismo: pega os melhores
        for elite_idx in range(amount_elite_ind):
            new_pop.append(population[POPULATION_NUMBER - elite_idx - 1])

        # 2) Soluções aleatórias da parte não-elite
        selected_bad = []
        for _ in range(amount_bad_ind):
            if amount_elite_ind >= POPULATION_NUMBER:
                break

            index = random_number(amount_elite_ind, POPULATION_NUMBER - 1)

            while index in selected_bad:
                index = random_number(amount_elite_ind, POPULATION_NUMBER - 1)

            selected_bad.append(index)
            new_pop.append(population[index])

        # 3) Filhos por crossover
        for _ in range(amount_new_ind):
            parent1 = choose_parent(population, 1)
            parent2 = choose_parent(population, 2)

            new_solution = crossover(parent1, parent2)
            evaluate_solution(new_solution)
            new_pop.append(new_solution)

        # Garante que a população não fique menor que o esperado
        while len(new_pop) < POPULATION_NUMBER:
            new_solution = create_solution()
            evaluate_solution(new_solution)
            new_pop.append(new_solution)

        # Garante que não passe do tamanho
        population = new_pop[:POPULATION_NUMBER]

        # Disaster
        if generation > 0 and generation % DISASTER_COUNTDOWN_EVERY == 0:
            population = disaster(population)

            if not population:
                raise ValueError(f"A população ficou vazia após disaster() na geração {generation}")

            for individual in population:
                evaluate_solution(individual)

        if not population:
            raise ValueError(f"A população ficou vazia na geração {generation}")

        population.sort()

        if best_solution_so_far.rank < population[-1].rank:
            best_solution_so_far = population[-1]

    print(f"\n----------------------")
    print(f"RANKING OF BEST SOLUTION: {best_solution_so_far.rank}")
    print(f"----------------------\n")

    return best_solution_so_far

In [23]:
print(main())

iteration 0
28987.82999999995
iteration 1

----------------------
RANKING OF BEST SOLUTION: 28987.82999999995
----------------------

